# Notebook 04 — Clasificación Random Forest, Diagnóstico Preliminar 70/30 y Mapas (Objetivo Específico 2)

- **Instancia SEPAL recomendada:** `m4` (4 CPU, 16 GB RAM, 0.19 USD/h) o `m8` (8 CPU, 32 GB RAM, 0.39 USD/h).
- **Responde al Objetivo Específico 2:** *"Estimar la superficie espacial del bosque manglar aplicando el modelo combinado MS+SAR."*
- **Flujo del notebook:**
  1. Entrena y ajusta 4 modelos Random Forest sobre el **70% de entrenamiento** con Validación Cruzada Estratificada (5-fold CV):
     - **M1 (MS):** Solo Sentinel-2 (20 bandas biestacionales).
     - **M2 (SAR):** Solo Sentinel-1 (4 bandas biestacionales).
     - **M3 (MS+SAR PCA):** Combinado con las bandas seleccionadas en el Objetivo 1.
     - **M4 (MS+SAR Completo):** Combinado con las 24 bandas (prueba de ablación).
  2. Evalúa el **30% preliminar** (OA, Kappa, PU, PP, **F1-Score Manglar**, importancia por permutación y McNemar preliminar) para confirmar que la base de entrenamiento es sólida antes de exportar mapas.
  3. Genera en GEE los **Mapas Clasificados a 10 m** (recortados estrictamente a `MASCARA_AOI_BASE == 1`), calcula la **superficie mapeada (ha)** y construye el **Mapa de Estratos** listo para importar directamente en **AREA2** y validar en **Collect Earth Online (CEO)**.

In [ ]:
# 1. Configuracion y carga de muestras extraidas + bandas seleccionadas por PCA
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from src.config_utils import (
    ensure_output_dirs,
    get_class_maps,
    get_predictor_band_names,
    init_gee,
    load_aoi_geometry,
    load_config,
)
from src.metrics import compute_preliminary_metrics, mcnemar_test
from src.plotting import plot_confusion_matrix, plot_model_comparison, set_thesis_style
from src.stack_builder import get_stack_asset_ids

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)
set_thesis_style()

reg_l = REGION.lower()
CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
CLASS_IDS = sorted(CLASES_NOMBRES.keys())
BANDS = get_predictor_band_names(cfg)
SEED = int(cfg["proyecto"]["seed"])

df_all = pd.read_csv(dirs["tablas"] / f"valores_24bandas_{reg_l}.csv")
pca_sel = json.loads((dirs["tablas"] / f"bandas_seleccionadas_pca_{reg_l}.json").read_text(encoding="utf-8"))

df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
df_test = df_all[df_all["split"] == "test_preliminar"].reset_index(drop=True)

MODEL_SPECS = {
    "MS": BANDS["s2_todas"],
    "SAR": BANDS["sar_todas"],
    "MS_SAR_PCA": pca_sel["bandas_seleccionadas"],
    "MS_SAR_FULL": BANDS["todas"],
}

print(f"Region: {REGION} | Train (70%): {len(df_train)} | Test preliminar (30%): {len(df_test)}")
for k, cols in MODEL_SPECS.items():
    print(f"  - {k:12s}: {len(cols):2d} variables")

In [ ]:
# 2. Entrenamiento Random Forest (5-fold CV en 70%) y diagnostico preliminar (30% con F1-Score)
rf_cfg = cfg["clasificacion"]["rf"]
cv = StratifiedKFold(n_splits=int(rf_cfg["cv_folds"]), shuffle=True, random_state=SEED)

modelos_entrenados = {}
preds_prelim = {}
metricas_prelim = {}
resumen_rows = []

y_train = df_train["clase_id"].astype(int).values
y_test = df_test["clase_id"].astype(int).values

for nombre_m, cols in MODEL_SPECS.items():
    X_tr = df_train[cols].values
    X_te = df_test[cols].values

    base_rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced_subsample")
    grid = GridSearchCV(
        base_rf,
        param_grid=rf_cfg["param_grid"],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X_tr, y_train)
    best_rf = grid.best_estimator_
    y_pred = best_rf.predict(X_te)

    res = compute_preliminary_metrics(y_test, y_pred, CLASS_IDS, CLASES_NOMBRES)
    mangle_row = res["per_class"].set_index("clase_id").loc[1]

    modelos_entrenados[nombre_m] = {"estimator": best_rf, "params": grid.best_params_, "cols": cols, "cv_f1_macro": grid.best_score_}
    preds_prelim[nombre_m] = y_pred
    metricas_prelim[nombre_m] = res

    # Guardar modelo local
    joblib.dump(modelos_entrenados[nombre_m], dirs["modelos"] / f"rf_{nombre_m.lower()}_{reg_l}.joblib")

    resumen_rows.append({
        "Modelo": nombre_m,
        "N_variables": len(cols),
        "CV_F1_Macro": grid.best_score_,
        "OA": res["OA"],
        "Kappa": res["Kappa"],
        "Macro_F1": res["Macro_F1"],
        "PU_Mangle": mangle_row["PU"],
        "PP_Mangle": mangle_row["PP"],
        "F1_Mangle": mangle_row["F1"],
        "Mejores_Params": json.dumps(grid.best_params_),
    })

df_resumen_prelim = pd.DataFrame(resumen_rows)
print("=== DIAGNOSTICO PRELIMINAR (SPLIT 70/30) ===")
display(df_resumen_prelim)

# Pruebas preliminares de McNemar (combinado vs simples)
for simple in ["MS", "SAR"]:
    for comb in ["MS_SAR_PCA", "MS_SAR_FULL"]:
        mc_multi = mcnemar_test(y_test, preds_prelim[comb], preds_prelim[simple])
        mc_mangle = mcnemar_test(y_test, preds_prelim[comb], preds_prelim[simple], binary_target_class=1)
        print(
            f"McNemar preliminar {comb} vs {simple} -> "
            f"Multiclase p={mc_multi['p_value_recomendado']:.4f} | "
            f"Binario Mangle p={mc_mangle['p_value_recomendado']:.4f}"
        )

df_resumen_prelim.to_csv(dirs["tablas"] / f"diagnostico_preliminar_70_30_{reg_l}.csv", index=False)

In [ ]:
# 3. Matrices de confusion preliminares e Importancia por Permutacion (contraste con PCA)
fig_comp = plot_model_comparison(
    df_resumen_prelim,
    f"Diagnóstico Preliminar (30% Control) — {cfg['regiones'][REGION]['nombre_largo']}",
    save_path=dirs["figuras"] / f"OE2_preliminar_comparativa_{reg_l}.png",
)
plt.show()

for nombre_m in ["MS", "SAR", "MS_SAR_PCA", "MS_SAR_FULL"]:
    fig_cm = plot_confusion_matrix(
        metricas_prelim[nombre_m]["confusion_matrix"],
        f"Matriz de Confusión Preliminar (30%) — {nombre_m} ({REGION})",
        is_proportion=False,
        save_path=dirs["figuras"] / f"OE2_prelim_cm_{nombre_m.lower()}_{reg_l}.png",
    )
    plt.show()

# Importancia por permutacion del modelo combinado completo (24 bandas)
rf_full = modelos_entrenados["MS_SAR_FULL"]["estimator"]
perm = permutation_importance(
    rf_full,
    df_test[BANDS["todas"]].values,
    y_test,
    scoring="f1_macro",
    n_repeats=30,
    random_state=SEED,
    n_jobs=-1,
)
df_perm = pd.DataFrame(
    {"importancia_perm": perm.importances_mean, "std_perm": perm.importances_std},
    index=BANDS["todas"],
).sort_values("importancia_perm", ascending=False)
display(df_perm.head(12))
df_perm.to_csv(dirs["tablas"] / f"importancia_permutacion_rf_{reg_l}.csv")

In [ ]:
# 4. Clasificacion espacial completa en GEE (10 m UTM, recortada a MASCARA_AOI_BASE == 1)
# y cuantificacion exacta de superficie mapeada (ha) por clase

stack_ids = get_stack_asset_ids(REGION, cfg)
stack_img = ee.Image(stack_ids["stack"])
qa_img = ee.Image(stack_ids["qa"])
aoi_base = load_aoi_geometry(cfg["assets"]["aoi"][REGION])
mascara_nucleo = qa_img.select("MASCARA_COMUN").eq(1).And(qa_img.select("MASCARA_AOI_BASE").eq(1))

# Preparar FeatureCollection de entrenamiento (70%) con las 24 bandas ya limpias
train_features = []
for _, row in df_train.iterrows():
    props = {"clase_id": int(row["clase_id"])}
    for b in BANDS["todas"]:
        props[b] = float(row[b])
    geom = ee.Geometry.Point([float(row["lon"]), float(row["lat"])])
    train_features.append(ee.Feature(geom, props))
fc_train_gee = ee.FeatureCollection(train_features)

def clasificar_en_gee(nombre_m: str, cols: list[str]) -> ee.Image:
    params = modelos_entrenados[nombre_m]["params"]
    n_trees = int(params.get("n_estimators", 300))
    min_leaf = int(params.get("min_samples_leaf", 2))
    max_feat = params.get("max_features", "sqrt")
    vars_per_split = int(max(1, round(np.sqrt(len(cols))))) if max_feat == "sqrt" else int(max(1, round(float(max_feat) * len(cols))))

    clf = ee.Classifier.smileRandomForest(
        numberOfTrees=n_trees,
        variablesPerSplit=vars_per_split,
        minLeafPopulation=min_leaf,
        seed=SEED,
    ).train(
        features=fc_train_gee,
        classProperty="clase_id",
        inputProperties=cols,
    )
    return (
        stack_img.select(cols)
        .classify(clf)
        .updateMask(mascara_nucleo)
        .rename("clasificacion")
        .toUint8()
    )

mapas_gee = {nombre_m: clasificar_en_gee(nombre_m, cols) for nombre_m, cols in MODEL_SPECS.items()}

# Mapa de estratos de Acuerdo/Desacuerdo para AREA2 (opcional y muy eficiente):
#   1: Mangle (Acuerdo total MS + SAR + MS_SAR_PCA)
#   2: Bosque latifoliado (MS_SAR_PCA, sin conflicto mangle)
#   3: Agua (MS_SAR_PCA, sin conflicto mangle)
#   4: Infraestructura urbana (MS_SAR_PCA, sin conflicto mangle)
#   5: Pastos/Ganaderia (MS_SAR_PCA, sin conflicto mangle)
#   6: Zona de desacuerdo en Mangle (alguno de MS/SAR/MS_SAR_PCA predice Mangle y otro no)
m_ms = mapas_gee["MS"].eq(1)
m_sar = mapas_gee["SAR"].eq(1)
m_comb = mapas_gee["MS_SAR_PCA"].eq(1)
suma_mangle = m_ms.add(m_sar).add(m_comb)
desacuerdo_mangle = suma_mangle.gt(0).And(suma_mangle.lt(3))
mapa_estratos_desacuerdo = mapas_gee["MS_SAR_PCA"].where(desacuerdo_mangle, 6).rename("estrato").toUint8()

def calcular_areas_ha(img: ee.Image, band_name: str, region_geom: ee.Geometry) -> dict[int, float]:
    area_img = ee.Image.pixelArea().divide(10000.0).addBands(img.select(band_name))
    grouped = area_img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
        geometry=region_geom,
        scale=10,
        maxPixels=int(cfg["gee"]["max_pixels"]),
        tileScale=8,
    ).get("groups").getInfo()
    return {int(g["clase"]): float(g["sum"]) for g in grouped}

areas_por_modelo = {}
filas_areas = []
for nombre_m, img_m in mapas_gee.items():
    a_dict = calcular_areas_ha(img_m, "clasificacion", aoi_base)
    areas_por_modelo[nombre_m] = a_dict
    fila = {"Modelo": nombre_m, "Total_AOI_ha": sum(a_dict.values())}
    for cid in CLASS_IDS:
        fila[f"{CLASES_CORTOS[cid]}_ha"] = a_dict.get(cid, 0.0)
    fila["Dif_vs_MARN2013_ha"] = a_dict.get(1, 0.0) - float(cfg["regiones"][REGION]["linea_base_marn_2013_ha"])
    filas_areas.append(fila)

areas_estratos_desac = calcular_areas_ha(mapa_estratos_desacuerdo, "estrato", aoi_base)

df_areas_oe2 = pd.DataFrame(filas_areas)
print(f"=== SUPERFICIE CARTOGRAFIADA (ha) — {REGION} (Linea base MARN 2013: {cfg['regiones'][REGION]['linea_base_marn_2013_ha']:,.2f} ha) ===")
display(df_areas_oe2)
print("Areas por estrato (Mapa Acuerdo/Desacuerdo 1..6) [ha]:", {k: round(v, 2) for k, v in sorted(areas_estratos_desac.items())})

# Guardar areas exactas en JSON para el Notebook 05 (Olofsson / Stehman)
payload_areas = {
    "region": REGION,
    "areas_modelos_ha": areas_por_modelo,
    "areas_estratos_acuerdo_desacuerdo_ha": areas_estratos_desac,
}
(dirs["tablas"] / f"areas_estratos_{reg_l}.json").write_text(json.dumps(payload_areas, indent=2), encoding="utf-8")
df_areas_oe2.to_csv(dirs["tablas"] / f"OE2_superficies_mapeadas_{reg_l}.csv", index=False)

In [ ]:
# 5. Inspeccion visual interactiva de los mapas clasificados
palette_5 = [CLASES_COLORES[c].lstrip("#") for c in CLASS_IDS]
palette_6 = palette_5 + ["984ea3"]  # morado = estrato 6 (desacuerdo en mangle)

m = geemap.Map(height="600px")
m.centerObject(aoi_base, 11)
m.add_basemap("SATELLITE")

m.addLayer(stack_img, {"bands": ["B8_SECA", "B11_SECA", "B4_SECA"], "min": 0, "max": 0.40}, "Stack Falsocolor (B8-B11-B4)", False)
m.addLayer(mapas_gee["MS"], {"min": 1, "max": 5, "palette": palette_5}, "Mapa M1: Solo MS (S2)", False)
m.addLayer(mapas_gee["SAR"], {"min": 1, "max": 5, "palette": palette_5}, "Mapa M2: Solo SAR (S1)", False)
m.addLayer(mapas_gee["MS_SAR_PCA"], {"min": 1, "max": 5, "palette": palette_5}, "Mapa M3: Combinado MS+SAR (PCA)", True)
m.addLayer(mapas_gee["MS_SAR_FULL"], {"min": 1, "max": 5, "palette": palette_5}, "Mapa M4: Combinado MS+SAR (24b)", False)
m.addLayer(mapa_estratos_desacuerdo, {"min": 1, "max": 6, "palette": palette_6}, "Estratos AREA2 (Acuerdo/Desacuerdo)", False)
m.add_legend(title="Coberturas", legend_dict={CLASES_NOMBRES[c]: CLASES_COLORES[c] for c in CLASS_IDS})
m

In [ ]:
# 6. Exportar mapas clasificados a GEE Assets (listos para AREA2) y Drive (GeoTIFF para ArcGIS/QGIS)
EJECUTAR_EXPORTS_MAPAS = False

reg_cfg = cfg["regiones"][REGION]
ver = cfg["proyecto"]["version_pipeline"].lower()
root_asset = cfg["gee"]["asset_root"].rstrip("/") + "/"
drive_folder = cfg["gee"]["drive_folder"]

if EJECUTAR_EXPORTS_MAPAS:
    export_items = {
        **{f"mapa_{k.lower()}_{reg_l}_{ver}": v for k, v in mapas_gee.items()},
        f"mapa_estratos_area2_{reg_l}_{ver}": mapa_estratos_desacuerdo,
    }
    for nombre, img_exp in export_items.items():
        t_asset = ee.batch.Export.image.toAsset(
            image=img_exp,
            description=f"Asset_{nombre}",
            assetId=root_asset + nombre,
            region=aoi_base,
            crs=reg_cfg["crs"],
            crsTransform=reg_cfg["crs_transform"],
            maxPixels=int(cfg["gee"]["max_pixels"]),
            pyramidingPolicy={".default": "mode"},
        )
        t_asset.start()

        t_drv = ee.batch.Export.image.toDrive(
            image=img_exp,
            description=f"GeoTIFF_{nombre}",
            folder=drive_folder,
            fileNamePrefix=nombre,
            region=aoi_base,
            crs=reg_cfg["crs"],
            crsTransform=reg_cfg["crs_transform"],
            maxPixels=int(cfg["gee"]["max_pixels"]),
            formatOptions={"cloudOptimized": True},
        )
        t_drv.start()
        print(f"[ENVIADO] Asset: {root_asset + nombre} | Drive COG: {nombre}.tif")
else:
    print("Modo seguro (EJECUTAR_EXPORTS_MAPAS = False). Aprueba el diagnostico 70/30 y el mapa visual antes de exportar.")